<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session12-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 12: in-class discussion problem (1 of 3)

This is the block 1 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then fix and run the code below, discuss the outcome with your neighbour, and answer the quiz again.

## Do contacts continue along the diagonal, or across it?

A contact map marks which residue pairs $(i, j)$ are close in space (here: $C_\beta$ atoms within
8 Å). Contacts are not independent pixels: when $i$ touches $j$, the neighbouring pairs are much more
likely to touch as well. *Which* neighbours depends on how the two stretches of chain run.

Two **antiparallel** β-strands pair up, and residue $i$ is in contact with residue $j$. Which
neighbouring pair is then most likely to be in contact too: $(i+1, j+1)$ along the diagonal of the
map, or $(i+1, j-1)$ across it?

**Fix and run.** The cell below downloads three all-β and three all-α structures from the PDB and
averages the 3 × 3 neighbourhood around every long-range contact. The last lines compare the two
corner pairs, but the line marked `FIX` reads the wrong corners of the 3 × 3 matrix `P` (rows are
$i-1, i, i+1$; columns are $j-1, j, j+1$). Run it once as it is, then fix it and run it again.

In [1]:
# Install the libraries Colab does not come with (does nothing if they are already installed)
import importlib.util, subprocess, sys
for module, package in [("Bio", "biopython")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

In [2]:
import io
import numpy as np, requests
from Bio.PDB import PDBParser

def neighbourhood(pid, chain_id):
    """The summed 3 x 3 neighbourhood of every contact (i, j) with j - i >= 12, and their number."""
    text = requests.get(f"https://files.rcsb.org/download/{pid}.pdb", timeout=60).text
    chain = PDBParser(QUIET=True).get_structure(pid, io.StringIO(text))[0][chain_id]
    res = [r for r in chain if r.id[0] == " " and "CA" in r]
    xyz = np.array([(r["CB"] if "CB" in r else r["CA"]).coord for r in res])
    C = np.linalg.norm(xyz[:, None] - xyz[None], axis=2) < 8       # the contact map
    acc, n = np.zeros((3, 3)), 0
    for a in range(1, len(C) - 1):
        for b in range(a + 12, len(C) - 1):                          # skip local turns
            if C[a, b]:
                acc += C[a - 1:a + 2, b - 1:b + 2]; n += 1
    return acc, n

classes = {"all-beta":  [("1TEN", "A"), ("2PTN", "A"), ("1TTG", "A")],
           "all-alpha": [("4HHB", "B"), ("1MBN", "A"), ("2HMQ", "A")]}
for name, proteins in classes.items():
    acc, n = np.zeros((3, 3)), 0
    for pid, chain_id in proteins:
        a, k = neighbourhood(pid, chain_id); acc += a; n += k
    P = acc / n                                  # P[r, c]: neighbour (i-1+r, j-1+c) is a contact too
    print(f"{name}: {n} contacts")
    print(np.round(P, 2))
    diagonal = (P[0, 2] + P[2, 0]) / 2           # FIX: these are not (i-1, j-1) and (i+1, j+1)
    across = (P[0, 0] + P[2, 2]) / 2
    print(f"  along the diagonal, (i+1, j+1): {diagonal:.2f}    across it, (i+1, j-1): {across:.2f}\n")

all-beta: 887 contacts
[[0.27 0.48 0.4 ]
 [0.47 1.   0.46]
 [0.41 0.48 0.27]]
  along the diagonal, (i+1, j+1): 0.40    across it, (i+1, j-1): 0.27

all-alpha: 451 contacts
[[0.1  0.24 0.09]
 [0.24 1.   0.24]
 [0.08 0.23 0.1 ]]
  along the diagonal, (i+1, j+1): 0.09    across it, (i+1, j-1): 0.10



**If you finish early.**

1. In the all-α proteins, the side neighbours $(i, j \pm 1)$ and $(i \pm 1, j)$ are in contact only
   about a quarter of the time, much less often than in the β-proteins. Why? (A helix turns every
   3.6 residues.)
2. For two *parallel* strands, which corner would you expect to win?
3. How could a predictor use these numbers to clean up a noisy coevolution map? (PconsC2, then the
   deep residual CNNs.)

**Discussion point.** As it was, the cell read the corners $(i-1, j+1)$ and $(i+1, j-1)$ as
"the diagonal", and so seemed to say that contacts continue along it. The fix:

```python
diagonal = (P[0, 0] + P[2, 2]) / 2
across = (P[0, 2] + P[2, 0]) / 2
```

In the all-β proteins, $(i+1, j-1)$ is then a contact 0.40 of the time, against 0.27 for
$(i+1, j+1)$. In two antiparallel strands the chains run in opposite directions: one step along the
first strand ($i \to i+1$) faces one step *back* along the second ($j \to j-1$). In the map, a
β-hairpin or antiparallel sheet is a line perpendicular to the diagonal; parallel strands give lines
parallel to it. Helices give a weaker, patchier neighbourhood, because only every third or fourth
residue faces the partner helix.

That is why contact predictors look at a whole patch of the map at once, as the convolutions of
Session 10 do, and why AlphaFold2 updates every pair from other pairs.